In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_57_try_4.pickle

In [ ]:
%%cudf.pandas.profile
### cell 58 ###

# Optimized cell for GPU via cudf.pandas

# Ensure question_of_interest_cell70 is defined
question_of_interest_cell70 = 'Do you use any of the following data storage products?'

# Step 1: Perform the column-name replacement on GPU
responses_df_2022_cell10.columns = (
    responses_df_2022_cell10.columns
      .to_series()
      .str.replace("(AWS/GCP/Azure customers only)", "for AWS GCP and Azure customers")
      .tolist()
)

# Step 2: Rewrite the subset function to use GPU string methods end-to-end
def grab_subset_of_data_70(original_df, question_of_interest):
    # GPU: turn the columns Index into a cudf.Series and mask by .str.contains
    col_series = original_df.columns.to_series()
    selected_cols = col_series[col_series.str.contains(question_of_interest)].index
    df = original_df[selected_cols]

    # GPU: split on '-' and strip leading whitespace to build new column names
    new_names = (
        df.columns
          .to_series()
          .str.split('-')
          .str.get(-1)
          .str.lstrip()
          .tolist()
    )
    df.columns = new_names

    # GPU: drop rows where all selected columns are null
    return df.dropna(how='all')

# Step 3: Apply the function and view the result
storage_df_2022 = grab_subset_of_data_70(
    responses_df_2022_cell10,
    question_of_interest_cell70
)
storage_df_2022.info()